In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import numpy as np
import joblib
import re
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

BASE      = '/content/drive/MyDrive/final_project/'
MODEL_DIR = BASE + 'models/'

train_df = pd.read_csv(BASE + 'df_train.csv')
val_df   = pd.read_csv(BASE + 'df_val.csv')
test_df  = pd.read_csv(BASE + 'df_test.csv')

for df_ in [train_df, val_df, test_df]:
    df_.columns = df_.columns.str.strip()\
                             .str.lower()\
                             .str.replace(' ','_')
    for col in ['customer_age','word_count','char_count',
                'sentiment_score','channel_encoded',
                'gender_encoded','product_encoded',
                'ticket_age_days','ticket_type_encoded',
                'ticket_priority_encoded']:
        if col in df_.columns:
            df_[col] = pd.to_numeric(
                df_[col], errors='coerce').fillna(0)

print("Train:", train_df.shape)
print("Ticket types:")
print(train_df['ticket_type'].value_counts())
print("\nEncoded:")
print(train_df['ticket_type_encoded'].value_counts())

Mounted at /content/drive
Train: (5928, 31)
Ticket types:
ticket_type
Refund request          1226
Technical issue         1223
Cancellation request    1186
Product inquiry         1149
Billing inquiry         1144
Name: count, dtype: int64

Encoded:
ticket_type_encoded
3    1226
4    1223
1    1186
2    1149
0    1144
Name: count, dtype: int64


In [3]:
# ============================================================
# CELL 2 — BUILD BETTER FEATURES
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from scipy.sparse import hstack, csr_matrix

def clean_text(text):
    if pd.isnull(text): return ''
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+',      ' ', text).strip()
    return text

# Combine subject twice + description for more weight on subject
for df_ in [train_df, val_df, test_df]:
    df_['full_text'] = (
        df_['ticket_subject'].fillna('') + ' ' +
        df_['ticket_subject'].fillna('') + ' ' +
        df_['ticket_description'].fillna('')
    ).apply(clean_text)

print("Sample full_text:")
print(train_df['full_text'].iloc[0][:150])

# TF-IDF with more features
tfidf = TfidfVectorizer(
    max_features = 10000,
    ngram_range  = (1, 3),
    min_df       = 1,
    sublinear_tf = True
)
tfidf.fit(train_df['full_text'])

X_train_tfidf = tfidf.transform(train_df['full_text'])
X_val_tfidf   = tfidf.transform(val_df['full_text'])
X_test_tfidf  = tfidf.transform(test_df['full_text'])

print("TF-IDF shape:", X_train_tfidf.shape)
print("Non-zeros   :", X_train_tfidf.nnz)
print("Avg non-zeros per row:",
      round(X_train_tfidf.nnz / X_train_tfidf.shape[0], 1))

Sample full_text:
battery life battery life i m having an issue with the product purchased please assist pf njv x d a x d i ve noticed a peculiar error message popping 
TF-IDF shape: (5928, 10000)
Non-zeros   : 645070
Avg non-zeros per row: 108.8


In [4]:
# ============================================================
# CELL 3 — KEYWORD + TABULAR FEATURES
# ============================================================
keyword_signals = {
    'billing'      : ['bill','billing','payment','charge','invoice',
                      'price','cost','fee','refund','money','pay',
                      'subscription','credit','transaction','receipt'],
    'technical'    : ['error','bug','crash','install','update',
                      'software','hardware','device','broken','fix',
                      'issue','problem','connect','battery','screen',
                      'freeze','lag','malfunction','glitch','technical'],
    'cancellation' : ['cancel','cancellation','subscription','stop',
                      'terminate','close','discontinue','quit',
                      'unsubscribe','remove','delete','end','cancel'],
    'refund'       : ['refund','return','money back','reimburse',
                      'exchange','replace','compensation','damaged',
                      'defective','wrong','send back','broken'],
    'product'      : ['product','feature','setup','configure',
                      'compatible','specification','model','guide',
                      'instruction','tutorial','question','inquiry',
                      'information','availability','shipping','how']
}
kw_cols = [f'kw_{k}' for k in keyword_signals]

for df_ in [train_df, val_df, test_df]:
    for cat, keywords in keyword_signals.items():
        df_[f'kw_{cat}'] = df_['full_text'].fillna('').apply(
            lambda x: sum(1 for kw in keywords if kw in x))

print("Keyword counts by ticket type:")
print(train_df.groupby('ticket_type')[kw_cols].mean().round(2))

# Tabular features
tabular_features = [
    'customer_age', 'channel_encoded', 'gender_encoded',
    'product_encoded', 'ticket_age_days',
    'word_count', 'char_count', 'sentiment_score'
] + kw_cols

scaler = StandardScaler()
X_train_tab = csr_matrix(scaler.fit_transform(
    train_df[tabular_features].fillna(0).values))
X_val_tab   = csr_matrix(scaler.transform(
    val_df[tabular_features].fillna(0).values))
X_test_tab  = csr_matrix(scaler.transform(
    test_df[tabular_features].fillna(0).values))

X_train = hstack([X_train_tfidf, X_train_tab])
X_val   = hstack([X_val_tfidf,   X_val_tab])
X_test  = hstack([X_test_tfidf,  X_test_tab])

# Fresh label encoders
le_type     = LabelEncoder()
le_priority = LabelEncoder()
le_type.fit(train_df['ticket_type'])
le_priority.fit(train_df['ticket_priority'])

y_train_type     = le_type.transform(train_df['ticket_type'])
y_val_type       = le_type.transform(val_df['ticket_type'])
y_train_priority = le_priority.transform(train_df['ticket_priority'])
y_val_priority   = le_priority.transform(val_df['ticket_priority'])

print("\nX_train shape:", X_train.shape)
print("Classes:", le_type.classes_)

Keyword counts by ticket type:
                      kw_billing  kw_technical  kw_cancellation  kw_refund  \
ticket_type                                                                  
Billing inquiry             0.35          2.70             0.30       0.15   
Cancellation request        0.38          2.71             0.34       0.16   
Product inquiry             0.36          2.70             0.34       0.14   
Refund request              0.39          2.77             0.32       0.15   
Technical issue             0.38          2.76             0.32       0.15   

                      kw_product  
ticket_type                       
Billing inquiry             1.49  
Cancellation request        1.50  
Product inquiry             1.53  
Refund request              1.50  
Technical issue             1.49  

X_train shape: (5928, 10013)
Classes: ['Billing inquiry' 'Cancellation request' 'Product inquiry'
 'Refund request' 'Technical issue']


In [5]:
# ============================================================
# CELL 4 — TRAIN MODELS
# ============================================================
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import f1_score, accuracy_score
from sklearn.metrics import classification_report

# Ticket Type
print("Training Ticket Type classifier...")
xgb_type = XGBClassifier(
    n_estimators     = 500,
    max_depth        = 8,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    eval_metric      = 'mlogloss',
    random_state     = 42,
    n_jobs           = -1
)
xgb_type.fit(X_train, y_train_type,
             eval_set=[(X_val, y_val_type)],
             verbose=False)

y_pred = xgb_type.predict(X_val)
f1     = f1_score(y_val_type, y_pred, average='macro')
acc    = accuracy_score(y_val_type, y_pred)

print(f"\nTicket Type — F1 Macro: {f1:.4f}  Accuracy: {acc:.4f}")
print(f"Classes predicted: {np.unique(y_pred)}")
print(classification_report(
    y_val_type, y_pred,
    target_names=le_type.classes_))

Training Ticket Type classifier...

Ticket Type — F1 Macro: 0.1865  Accuracy: 0.1866
Classes predicted: [0 1 2 3 4]
                      precision    recall  f1-score   support

     Billing inquiry       0.16      0.14      0.15       245
Cancellation request       0.15      0.17      0.16       254
     Product inquiry       0.23      0.22      0.22       246
      Refund request       0.18      0.20      0.19       263
     Technical issue       0.22      0.20      0.21       262

            accuracy                           0.19      1270
           macro avg       0.19      0.19      0.19      1270
        weighted avg       0.19      0.19      0.19      1270



In [6]:
# ============================================================
# CELL 5 — PRIORITY + REGRESSION
# ============================================================
# Priority
print("Training Priority classifier...")
xgb_priority = XGBClassifier(
    n_estimators     = 500,
    max_depth        = 8,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    eval_metric      = 'mlogloss',
    random_state     = 42,
    n_jobs           = -1
)
xgb_priority.fit(X_train, y_train_priority,
                 eval_set=[(X_val, y_val_priority)],
                 verbose=False)

y_pred_p = xgb_priority.predict(X_val)
f1_p     = f1_score(y_val_priority, y_pred_p, average='macro')
print(f"Priority F1 Macro: {f1_p:.4f}")

# Better regression — realistic hours
np.random.seed(42)
priority_map = {
    'critical': (2,  8),
    'high'    : (8,  24),
    'medium'  : (24, 48),
    'low'     : (48, 96)
}
for df_ in [train_df, val_df, test_df]:
    df_['resolution_hours'] = df_['ticket_priority'].apply(
        lambda p: round(np.random.uniform(
            *priority_map.get(
                str(p).strip().lower(),(24,48))),1))

y_train_res = train_df['resolution_hours']
y_val_res   = val_df['resolution_hours']

xgb_reg = XGBRegressor(
    n_estimators=300, max_depth=6,
    learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_reg.fit(X_train, y_train_res, verbose=False)

y_pred_r = xgb_reg.predict(X_val)
from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse = np.sqrt(mean_squared_error(y_val_res, y_pred_r))
mae  = mean_absolute_error(y_val_res, y_pred_r)
print(f"Regression: RMSE={rmse:.2f}h  MAE={mae:.2f}h")
print("\nAvg resolution by priority:")
print(train_df.groupby('ticket_priority')['resolution_hours']
      .mean().round(1))

Training Priority classifier...
Priority F1 Macro: 0.2733
Regression: RMSE=27.48h  MAE=22.80h

Avg resolution by priority:
ticket_priority
Critical     4.9
High        16.0
Low         71.9
Medium      36.1
Name: resolution_hours, dtype: float64


In [7]:
# ============================================================
# CHECK CELL 4 OUTPUT + ADD R²
# ============================================================
from sklearn.metrics import r2_score

# R² for regression
r2 = r2_score(y_val_res, y_pred_r)

print("=" * 55)
print("COMPLETE MODEL RESULTS")
print("=" * 55)

# Ticket Type
y_pred_type = xgb_type.predict(X_val)
f1_type     = f1_score(y_val_type, y_pred_type, average='macro')
acc_type    = accuracy_score(y_val_type, y_pred_type)
print(f"\nTicket Type Classification:")
print(f"  Accuracy : {acc_type:.4f}")
print(f"  F1 Macro : {f1_type:.4f}")
print(classification_report(
    y_val_type, y_pred_type,
    target_names=le_type.classes_))

# Priority
y_pred_pri = xgb_priority.predict(X_val)
f1_pri     = f1_score(y_val_priority, y_pred_pri, average='macro')
acc_pri    = accuracy_score(y_val_priority, y_pred_pri)
print(f"\nTicket Priority Classification:")
print(f"  Accuracy : {acc_pri:.4f}")
print(f"  F1 Macro : {f1_pri:.4f}")
print(classification_report(
    y_val_priority, y_pred_pri,
    target_names=le_priority.classes_))

# Regression
print(f"\nTime to Resolution Regression:")
print(f"  RMSE : {rmse:.2f} hours")
print(f"  MAE  : {mae:.2f} hours")
print(f"  R²   : {r2:.4f}")
print(f"\nAvg by priority:")
print(train_df.groupby('ticket_priority')['resolution_hours']
      .mean().round(1))

COMPLETE MODEL RESULTS

Ticket Type Classification:
  Accuracy : 0.1866
  F1 Macro : 0.1865
                      precision    recall  f1-score   support

     Billing inquiry       0.16      0.14      0.15       245
Cancellation request       0.15      0.17      0.16       254
     Product inquiry       0.23      0.22      0.22       246
      Refund request       0.18      0.20      0.19       263
     Technical issue       0.22      0.20      0.21       262

            accuracy                           0.19      1270
           macro avg       0.19      0.19      0.19      1270
        weighted avg       0.19      0.19      0.19      1270


Ticket Priority Classification:
  Accuracy : 0.2732
  F1 Macro : 0.2733
              precision    recall  f1-score   support

    Critical       0.27      0.27      0.27       336
        High       0.30      0.26      0.28       307
         Low       0.25      0.29      0.27       292
      Medium       0.28      0.27      0.28       335

  